In [1]:
import os
import json
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score

os.makedirs('artifacts', exist_ok=True)
os.makedirs('evaluation', exist_ok=True)

df = pd.read_csv('experiments/data/cleaned_train.csv')
df = df.dropna(subset=['cleaned_text'])

categories = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
X = df['cleaned_text']
y = df[categories]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

models = {
    "Multinomial Naive Bayes": OneVsRestClassifier(MultinomialNB()),
    "Logistic Regression": OneVsRestClassifier(LogisticRegression(max_iter=1000, random_state=42)),
    "Linear SVM (Calibrated)": OneVsRestClassifier(CalibratedClassifierCV(LinearSVC(random_state=42, dual=False)))
}

best_f1 = 0.0
best_model_name = None
best_model_obj = None
results = {}

for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)
    
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='micro')
    
    results[name] = {
        "accuracy": float(acc),
        "f1_micro": float(f1)
    }
    
    if f1 > best_f1:
        best_f1 = f1
        best_model_name = name
        best_model_obj = model

with open('evaluation/multilabel_metrics.json', 'w') as f:
    json.dump(results, f, indent=4)

y_pred_best = best_model_obj.predict(X_test_tfidf)
report = classification_report(y_test, y_pred_best, target_names=categories)
with open('evaluation/multilabel_report.txt', 'w') as f:
    f.write(report)

joblib.dump(best_model_obj, 'artifacts/multilabel_toxic_model.pkl')
joblib.dump(vectorizer, 'artifacts/tfidf_vectorizer.pkl')
joblib.dump(categories, 'artifacts/categories.pkl')

print(f"Best Model: {best_model_name}")
print(f"F1-Score (Micro): {best_f1:.4f}")

c:\Users\Aayush Chauhan\SafeSpeak\backend\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Aayush Chauhan\SafeSpeak\backend\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Aayush Chauhan\SafeSpeak\backend\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric

Best Model: Linear SVM (Calibrated)
F1-Score (Micro): 0.7509
